# Project 03 (final) — Customer segmentation

**Module 05 — Machine Learning 2 · final project (unsupervised learning)**

This is the **final project** of the module. Unlike the first two projects there is **no prescribed code** here — you apply everything yourself: preprocessing, dimensionality reduction, four clustering methods, model selection and interpretation. Only the plain data infrastructure (the download) is given, so that you can start straight away.

Work through the sections in order. Each one comes with an **empty code cell** (your code) and after it a **reflection question** that you answer in writing. Only compare with `solution/solution.ipynb` at the very end.

---

### The task

A Portuguese wholesale distributor knows its 440 customers only through their **annual spending in six product categories** (`Fresh`, `Milk`, `Grocery`, `Frozen`, `Detergents_Paper`, `Delicassen`). Your task:

> **Find natural customer segments in a data-driven way. Answer: how many are there, what do they look like, which method describes them best — and how do you convince yourself (and the distributor) that the segments are real?**

**Two columns are off limits for the clustering:** `Channel` (Horeca vs. retail) and `Region`. You keep those for **external validation**: a good clustering obtained purely from the spending should rediscover the `Channel` structure *without ever having seen it*. That is exactly what you measure at the end with the adjusted Rand index.

**Why this format (a notebook):** segmentation lives on the interplay of plot, metric and interpretation side by side.
**Why real data (UCI Wholesale Customers):** compact (440×6), but with all the real pitfalls — strong right skew, overlapping clusters, an external truth to validate against. Ideal for pitting the methods of the module *against each other*.

## Setup (given)

Downloads the data into `datasets/` (cached, excluded via `.gitignore`) and separates the features from the external labels. **From here on you write it yourself.**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import urllib.request

RNG = 42
np.random.seed(RNG)

DATA_DIR = Path("datasets"); DATA_DIR.mkdir(exist_ok=True)
CSV = DATA_DIR / "wholesale_customers.csv"
URL = ("https://archive.ics.uci.edu/ml/machine-learning-databases/"
       "00292/Wholesale%20customers%20data.csv")
if not CSV.exists():
    req = urllib.request.Request(URL, headers={"User-Agent": "Mozilla/5.0"})
    CSV.write_bytes(urllib.request.urlopen(req, timeout=60).read())

df = pd.read_csv(CSV)
FEATURES = ["Fresh", "Milk", "Grocery", "Frozen", "Detergents_Paper", "Delicassen"]

# Keep the external labels safe (do NOT use them for the clustering!)
channel = df["Channel"].values - 1     # 0 = Horeca, 1 = retail
region  = df["Region"].values
X = df[FEATURES].values.astype(float)
print("X:", X.shape, "| channel distribution:", np.bincount(channel))
df.head()

## Task 1 — EDA and preprocessing

Look at the distributions of the six features. Explain **why clustering on the raw data would fail**, and derive your preprocessing from that.

**To do:**
- Histograms (or boxplots) of the six features; compute the skewness (`scipy.stats.skew`).
- Transform sensibly (keyword: multiplicative, log-normal quantities) and standardize afterwards.
- Create the preprocessed array (e.g. `Xs`) — everything that follows runs on it.

In [ ]:
# your code


> **Reflection 1:** Why is a log transformation important *before* the standardization here? What would happen with k-means if you clustered on the raw data? *(Answer:)*

## Task 2 — Dimensionality reduction and visualization

**To do:**
- **PCA** on `Xs`. A scree plot (how much variance is in PC1+PC2?). Produce a **biplot** and colour the points by the *held-back* `channel`.
- Interpret the **loadings** of PC1 and PC2: which product categories belong together, and what does that mean substantively?
- Optional: **t-SNE** (`perplexity≈30`) for the visualization. What does the cluster separation look like — sharp gaps or a smooth transition?

In [ ]:
# your code


> **Reflection 2:** What do the PC loadings tell you about the *kind* of customers? Why may you **not** interpret t-SNE distances the way you interpret Euclidean distances? *(Answer:)*

## Task 3 — Four clustering methods

Apply **all four** methods of the module and choose the hyperparameters of each one *with a clean justification* (do not guess):

1. **k-means** — determine $k$ via the **elbow** *and* the **silhouette**.
2. **Gaussian mixture (EM)** — choose the number of components **and** the covariance type via the **BIC**. Also look at the soft assignments (`predict_proba`).
3. **DBSCAN** — `min_samples` by the heuristic ($\approx 2\cdot\dim$), `eps` via the **k-distance plot**. What happens? Do not let the result surprise you — *explain* it.
4. **Agglomerative (Ward)** — draw the **dendrogram** and read off the natural number of clusters.

In [ ]:
# your code — k-means (elbow + silhouette)


In [ ]:
# your code — GMM/EM (BIC over components and covariance type)


In [ ]:
# your code — DBSCAN (k-distance plot, eps sweep)


In [ ]:
# your code — Ward (dendrogram)


> **Reflection 3:** Which number of clusters do the three "compact" methods (k-means, GMM, Ward) agree on? And why does DBSCAN give a completely different picture — which of its underlying assumptions is violated by this data? *(Answer:)*

## Task 4 — Validation: internal **and** external

Build a **comparison table** over your best solutions of all four methods, containing:
- **internal** measures: silhouette (↑), Davies-Bouldin (↓), Calinski-Harabasz (↑),
- the **external** measure: the **adjusted Rand index** against the held-back `channel`.

Careful: the internal optimum (e.g. the best BIC) and the external optimum (the best ARI) do **not** have to be the same method or the same number of clusters — that is part of the insight.

In [ ]:
# your code — comparison table (silhouette/DB/CH + ARI against channel)


> **Reflection 4:** Which method discovers the `Channel` structure most faithfully (the highest ARI) — and *why* does precisely its model assumption match this shape of data? Why is a single quality measure never enough? *(Answer:)*

## Task 5 — Name the segments and give a business recommendation

Take your **winning solution** and make it usable for the business:
- Describe every segment by its **median spending per category** (back in money space, not in z-scores).
- Give every segment a **telling name** and a short characterization.
- Show the PCA map twice side by side: on the left your discovered segments, on the right the true `channel`.
- Formulate **one concrete recommended action** for the distributor (logistics / assortment / marketing).

In [ ]:
# your code — segment profiles + final visualization


> **Reflection 5 (closing):** Summarize in 4–5 sentences: how many segments, which method, validated how, which business recommendation? And the meta-insight: in unsupervised learning, what determines *which* answer you get? *(Answer:)*

---
## Acceptance criteria (how you know you are done)

- [ ] The preprocessing is justified (log + z-score), the raw and the transformed picture are both shown.
- [ ] The PCA loadings are interpreted substantively; a 2D visualization is present.
- [ ] All **four** methods applied, every hyperparameter chosen **with a justification** (elbow/silhouette, BIC, k-distance, dendrogram).
- [ ] DBSCAN's behaviour is **explained** (not only reported).
- [ ] A comparison table with internal measures **and** the ARI against `channel`.
- [ ] Segments named, median profiles shown, one business recommendation formulated.
- [ ] All five reflection questions answered in writing.

**Reference values** (for rough orientation — small deviations are normal): k-means/GMM/Ward converge on about **2** main segments; the best ARI against `Channel` is about **0.6** (reached by the GMM with full covariance); DBSCAN finds **no** meaningful structure.